# 03 – TD Agents Experiment

**Flow:**
1. Setup & load data
2. Load MC Agent (đã train ở notebook 02)
3. Phase 1 – Train 4 TD Agents trên train stream
4. Warm-up – Căn chỉnh tất cả models trước khi evaluate
5. Phase 2 – Evaluate công bằng trên test stream
   - Static / Periodic / Always (baselines)
   - MC Agent (loaded từ pkl)
   - SARSA / Expected SARSA / Q-Learning / Double Q-Learning
6. Summary table & visualizations


## 1. Setup

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import copy
from collections import deque

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from config import (
    BATCH_SIZE, CATEGORICAL_COLS,
    FULL_WINDOW_BATCHES, PARTIAL_WINDOW_BATCHES,
    TD_EPISODE_STARTS,
)
from utils.data_loader import AirlinesDataLoader
from models.LightGBM import LightGBM
from metrics.stream_metrics import StreamMetrics
from environment.drift_env import DriftStreamEnv
from agents.mc_agent import MCAgent
from agents.td_agent import SARSAAgent, ExpectedSARSAAgent, QLearningAgent, DoubleQLearningAgent
from utils.visualizer import (
    plot_dashboard,
    plot_cumulative_reward,
    plot_action_distribution,
    plot_agent_vs_baselines,
    plot_retrain_events,
)

MC_AGENT_PATH = os.path.join(PROJECT_ROOT, 'experiments', 'saved_models', 'mc_agent.pkl')
TD_SAVE_DIR   = os.path.join(PROJECT_ROOT, 'experiments', 'saved_models')

print('Setup complete!')


## 2. Load Data

In [ ]:
loader = AirlinesDataLoader()
X_train, y_train = loader.get_initial_train_data()

print(f'Initial train size : {len(X_train):,}')
print(f'Total stream       : {loader.n_batches} batches')
print(f'Train stream       : {loader.n_train_batches} batches  (batch 0 – {loader.n_train_batches-1})')
print(f'Test  stream       : {loader.n_test_batches} batches  (batch {loader.n_train_batches} – {loader.n_batches-1})')
print(f'Warm-up window     : {FULL_WINDOW_BATCHES} batches  (cuối train stream)')


## 3. Load MC Agent

MC Agent đã được train ở notebook 02, load từ pkl để tái sử dụng cho phase evaluate.
Không train lại.


In [ ]:
agent_mc = MCAgent.load(MC_AGENT_PATH)
print(f'\nStates visited      : {len(agent_mc.Q)}')
print(f'Preferences learned : {len(agent_mc.H)}')
print(f'Final temperature   : {agent_mc.temp:.4f}')


## 4. Phase 1 – Train TD Agents

4 TD agents được train **chỉ trên train stream** (70% đầu).  
`env.reset(mode='train')` → `stream_train_batches()`.

Epsilon cố định = 0.1 trong suốt training (không decay).  
Khi evaluate: set `agent.epsilon = 0.0` → pure greedy policy.


In [ ]:
clf     = LightGBM()
metrics = StreamMetrics()
env     = DriftStreamEnv(loader, clf, metrics)

N_TRAIN_EPISODES = 200

agents_td = {
    'SARSA'          : SARSAAgent(),
    'Expected SARSA' : ExpectedSARSAAgent(),
    'Q-Learning'     : QLearningAgent(),
    'Double Q-Learning': DoubleQLearningAgent(),
}

td_histories = {}
for name, agent in agents_td.items():
    print(f'\n{("="*60)}')
    print(f'Training {name}...')
    history = agent.train(env, n_episodes=N_TRAIN_EPISODES, verbose=True)
    td_histories[name] = history
    print(f'  States visited: {len(agent.Q)}')


### Save TD Agents

In [ ]:
for name, agent in agents_td.items():
    fname = name.lower().replace(' ', '_').replace('-', '_') + '.pkl'
    path  = os.path.join(TD_SAVE_DIR, fname)
    agent.save(path)
    print(f'Saved {name} → {path}')


### Learning Curves – TD Agents (Training Phase)

In [ ]:
plot_cumulative_reward(
    {name: h['episode_rewards'] for name, h in td_histories.items()},
    title    = 'TD Agents – Episode Reward over Training (Train Stream)',
    filename = 'td_learning_curves.png',
)

for name, h in td_histories.items():
    rewards = h['episode_rewards']
    print(f'{name:<22} | Avg(all): {np.mean(rewards):.4f} | '
          f'Avg(last10): {np.mean(rewards[-10:]):.4f} | '
          f'Best: {max(rewards):.4f}')


## 5. Warm-up – Căn chỉnh Models trước khi Evaluate

Tất cả models (baselines, MC Agent, TD Agents) bắt đầu test stream
với classifier được warm-up trên `FULL_WINDOW_BATCHES` batches cuối train stream.


In [ ]:
X_warm, y_warm = loader.get_warmup_data(n_batches=FULL_WINDOW_BATCHES)
print(f'Warm-up data shape : {X_warm.shape}')
print(f'({FULL_WINDOW_BATCHES} batches × {BATCH_SIZE} samples = {FULL_WINDOW_BATCHES*BATCH_SIZE:,} samples)')

def warmup_clf(name: str) -> LightGBM:
    """Tạo fresh clf, train thẳng trên X_warm (FULL_WINDOW_BATCHES batches cuối train stream)."""
    warmed = LightGBM()
    warmed.full_retrain(X_warm, y_warm)
    print(f'  [{name}] Warmed up on {len(X_warm):,} samples')
    return warmed

print('Warming up all classifiers...')
clf_static        = warmup_clf('Static')
clf_periodic      = warmup_clf('Periodic')
clf_always        = warmup_clf('Always')
clf_warmed_agents = warmup_clf('Agents (MC + TD)')

print('\nAll models warmed up. Ready to evaluate on test stream.')

## 6. Phase 2 – Evaluate trên Test Stream

### Helper functions

In [ ]:
def evaluate_agent(agent, env: DriftStreamEnv, label: str,
                   warm_model: LightGBM, n_episodes: int = 3) -> dict:
    """
    Prequential evaluation của bất kỳ agent nào trên test stream.

    - mode='test' → stream_test_batches()
    - warm_model  → env.reset() dùng model đã warm-up thay vì train lại từ X_train
    - Không update agent (no learning)
    - Với TD agents: set epsilon = 0.0 trước khi evaluate → pure greedy
    - n_episodes > 1 để average qua stochasticity
    """
    print(f'\n=== Evaluating {label} on TEST stream ({n_episodes} episodes) ===')

    # Set epsilon = 0.0 nếu là TD agent → pure greedy khi evaluate
    original_epsilon = getattr(agent, 'epsilon', None)
    if original_epsilon is not None:
        agent.epsilon = 0.0
        print(f'  [TD] epsilon set to 0.0 for evaluation')

    all_preq_acc     = []
    all_rolling_err  = []
    all_drift        = []
    all_uncertainty  = []
    all_rewards      = []
    all_actions      = []
    all_retrain_events = []

    for ep in range(n_episodes):
        state     = env.reset(start_batch=0, mode='test', warm_model=warm_model)
        ep_reward = 0.0
        step      = 0
        ep_actions        = []
        ep_retrain_events = []

        while not env.is_done:
            # MC Agent dùng select_action trả về (action, pi)
            # TD Agent dùng select_action trả về action (int)
            result = agent.select_action(state)
            if isinstance(result, tuple):
                action, pi = result
            else:
                action, pi = result, None

            next_state, reward, done, info = env.step(
                action,current_state=state, pi=pi, update_explorer=False
            )

            if done and 'action_taken' not in info:
                break

            ep_reward += reward
            ep_actions.append(info['action_taken'])
            if info['action_taken'] in ['partial_update', 'full_retrain']:
                ep_retrain_events.append(info['batch_idx'])

            step  += 1
            state  = next_state

        all_rewards.append(ep_reward)
        all_actions.append(ep_actions)
        all_retrain_events.append(ep_retrain_events)
        all_preq_acc.append(env.metrics.prequential_acc_history.copy())
        all_rolling_err.append(env.metrics.rolling_error_history.copy())
        all_drift.append(env.metrics.drift_measure_history.copy())
        all_uncertainty.append(env.metrics.uncertainty_history.copy())
        print(f'  Episode {ep+1}: reward={ep_reward:.2f}, steps={step}')

    # Restore epsilon
    if original_epsilon is not None:
        agent.epsilon = original_epsilon

    # Average qua các episodes
    min_len         = min(len(h) for h in all_preq_acc)
    avg_preq_acc    = np.mean([h[:min_len] for h in all_preq_acc],    axis=0).tolist()
    avg_rolling_err = np.mean([h[:min_len] for h in all_rolling_err], axis=0).tolist()
    avg_drift       = np.mean([h[:min_len] for h in all_drift],       axis=0).tolist()
    avg_uncertainty = np.mean([h[:min_len] for h in all_uncertainty], axis=0).tolist()

    action_dist = {}
    for ep_actions_list in all_actions:
        for a in ep_actions_list:
            action_dist[a] = action_dist.get(a, 0) + 1
    for k in action_dist:
        action_dist[k] = round(action_dist[k] / n_episodes, 1)

    retrain_events = all_retrain_events[-1]

    print(f'\n  Avg reward     : {np.mean(all_rewards):.4f} ± {np.std(all_rewards):.4f}')
    print(f'  Final preq acc : {avg_preq_acc[-1]:.4f}')
    print(f'  Final roll err : {avg_rolling_err[-1]:.4f}')
    print(f'  Action dist    : {action_dist}')
    print(f'  Avg retrains   : {np.mean([len(e) for e in all_retrain_events]):.1f}')

    return {
        'label'          : label,
        'prequential_acc': avg_preq_acc,
        'rolling_error'  : avg_rolling_err,
        'drift_measure'  : avg_drift,
        'uncertainty'    : avg_uncertainty,
        'episode_rewards': all_rewards,
        'action_dist'    : action_dist,
        'retrain_events' : retrain_events,
        'avg_retrains'   : np.mean([len(e) for e in all_retrain_events]),
        'final_summary'  : {
            'prequential_accuracy': avg_preq_acc[-1],
            'rolling_error'       : avg_rolling_err[-1],
            'n_retrains'          : len(retrain_events),
            'avg_reward'          : np.mean(all_rewards),
        },
    }


def run_baseline_test(strategy: str, clf_warmed: LightGBM,
                      retrain_every: int = 50) -> dict:
    """Chạy baseline strategy trên TEST stream."""
    print(f'\n=== Baseline: {strategy.upper()} (test stream) ===')

    clf           = copy.deepcopy(clf_warmed)
    metrics_bl    = StreamMetrics()
    window_buffer = deque(maxlen=FULL_WINDOW_BATCHES)
    retrain_events = []

    # Set reference từ warm-up data
    fold_size   = len(X_warm) // 5
    init_errors = [
        clf.get_error_rate(
            X_warm.iloc[i*fold_size:(i+1)*fold_size],
            y_warm.iloc[i*fold_size:(i+1)*fold_size]
        ) for i in range(5)
    ]
    metrics_bl.set_reference(init_errors)

    batch_count = 0

    for X_batch, y_batch, idx in loader.stream_test_batches():
        window_buffer.append((X_batch, y_batch))

        # Prequential: predict TRƯỚC, retrain SAU
        preds      = clf.predict(X_batch)
        y_proba    = clf.predict_proba(X_batch)
        error_rate = clf.get_error_rate(X_batch, y_batch)
        metrics_bl.update(y_batch.values, preds, error_rate, y_proba)

        if strategy == 'periodic' and batch_count > 0 and batch_count % retrain_every == 0:
            recent = list(window_buffer)
            X_w = pd.concat([b[0] for b in recent], ignore_index=True)
            y_w = pd.concat([b[1] for b in recent], ignore_index=True)
            for col in CATEGORICAL_COLS:
                X_w[col] = X_w[col].astype('category')
            clf.full_retrain(X_w, y_w)
            retrain_events.append(idx)

        elif strategy == 'always':
            recent = list(window_buffer)
            X_w = pd.concat([b[0] for b in recent], ignore_index=True)
            y_w = pd.concat([b[1] for b in recent], ignore_index=True)
            for col in CATEGORICAL_COLS:
                X_w[col] = X_w[col].astype('category')
            clf.full_retrain(X_w, y_w)
            retrain_events.append(idx)

        batch_count += 1

    print(f'  Done! Batches: {batch_count}, Retrains: {len(retrain_events)}')
    return {
        'label'          : strategy,
        'prequential_acc': metrics_bl.prequential_acc_history,
        'rolling_error'  : metrics_bl.rolling_error_history,
        'drift_measure'  : metrics_bl.drift_measure_history,
        'uncertainty'    : metrics_bl.uncertainty_history,
        'retrain_events' : retrain_events,
        'avg_retrains'   : len(retrain_events),
        'final_summary'  : metrics_bl.summary(),
    }


### 6a. Baselines

In [ ]:
results_static   = run_baseline_test('static',   clf_static)
results_periodic = run_baseline_test('periodic', clf_periodic, retrain_every=50)
results_always   = run_baseline_test('always',   clf_always)


### 6b. MC Agent

In [ ]:
results_mc = evaluate_agent(
    agent_mc, env,
    label       = 'MC Agent',
    warm_model  = clf_warmed_agents,
    n_episodes  = 3,
)


### 6c. TD Agents

In [ ]:
results_td = {}
for name, agent in agents_td.items():
    results_td[name] = evaluate_agent(
        agent, env,
        label      = name,
        warm_model = clf_warmed_agents,
        n_episodes = 3,
    )


## 7. Summary Table

In [ ]:
all_results = {
    'Static'           : results_static,
    'Periodic'         : results_periodic,
    'Always'           : results_always,
    'MC Agent'         : results_mc,
    **{name: results_td[name] for name in agents_td},
}

print('\n' + '='*80)
print(f"{'Strategy':<24} {'Preq Acc':>10} {'Roll Err':>10} {'Avg Retrains':>14} {'Avg Reward':>12}")
print('='*80)

baseline_names = {'Static', 'Periodic', 'Always'}
for name, results in all_results.items():
    preq     = results['final_summary']['prequential_accuracy']
    err      = results['final_summary']['rolling_error']
    n_rt     = results['avg_retrains']
    avg_rew  = results['final_summary'].get('avg_reward', float('nan'))
    marker   = '' if name in baseline_names else ' ←'
    n_rt_str = f'{n_rt:.1f}' if isinstance(n_rt, float) else str(n_rt)
    print(f"{name:<24} {preq:>10.4f} {err:>10.4f} {n_rt_str:>14} {avg_rew:>12.4f}{marker}")

print('='*80)
print('* Evaluate trên test stream (30% cuối), warm-up điểm khởi đầu công bằng.')
print('* Avg Reward: N/A cho baselines (không có reward signal).')


## 8. Learning Curves – Tất cả Agents (Training Phase)

In [ ]:
all_train_histories = {
    'MC Agent': td_histories.get('MC Agent', {}).get('episode_rewards', []),
    **{name: td_histories[name]['episode_rewards'] for name in agents_td},
}
# Bỏ key nếu empty
all_train_histories = {k: v for k, v in all_train_histories.items() if v}

plot_cumulative_reward(
    all_train_histories,
    title    = 'All Agents – Episode Reward over Training (Train Stream)',
    filename = 'all_agents_learning_curves.png',
)


## 9. Prequential Accuracy – Tất cả Agents vs Baselines (Test Stream)

In [ ]:
baselines_acc = {
    'Static'  : results_static['prequential_acc'],
    'Periodic': results_periodic['prequential_acc'],
    'Always'  : results_always['prequential_acc'],
}
agents_acc = {
    'MC Agent': results_mc['prequential_acc'],
    **{name: results_td[name]['prequential_acc'] for name in agents_td},
}

plot_agent_vs_baselines(
    baselines = baselines_acc,
    agents    = agents_acc,
    ylabel    = 'Prequential Accuracy',
    title     = 'All Agents vs Baselines – Prequential Accuracy (Test Stream)',
    filename  = 'all_agents_vs_baselines_acc.png',
)


## 10. Rolling Error Rate – Tất cả Agents vs Baselines (Test Stream)

In [ ]:
baselines_err = {
    'Static'  : results_static['rolling_error'],
    'Periodic': results_periodic['rolling_error'],
    'Always'  : results_always['rolling_error'],
}
agents_err = {
    'MC Agent': results_mc['rolling_error'],
    **{name: results_td[name]['rolling_error'] for name in agents_td},
}

plot_agent_vs_baselines(
    baselines = baselines_err,
    agents    = agents_err,
    ylabel    = 'Rolling Error Rate',
    title     = 'All Agents vs Baselines – Rolling Error Rate (Test Stream)',
    filename  = 'all_agents_vs_baselines_err.png',
)


## 11. Action Distribution – Tất cả Agents (Test Stream)

In [ ]:
all_action_dists = {
    'MC Agent': results_mc['action_dist'],
    **{name: results_td[name]['action_dist'] for name in agents_td},
}

plot_action_distribution(
    all_action_dists,
    title    = 'All Agents – Action Distribution (Test Stream)',
    filename = 'all_agents_action_dist.png',
)

print('\nAction distribution (averaged over 3 eval episodes):')
for agent_name, dist in all_action_dists.items():
    total = sum(dist.values())
    print(f'\n  {agent_name}:')
    for action, count in sorted(dist.items(), key=lambda x: -x[1]):
        print(f'    {action:<16}: {count:>5}  ({100*count/total:.1f}%)')


## 12. TD Agent Dashboards (Test Stream)

In [ ]:
for name, results in results_td.items():
    fname = 'td_' + name.lower().replace(' ', '_').replace('-', '_') + '_dashboard.png'
    plot_dashboard(
        results,
        title    = f'{name} – Stream Metrics Dashboard (Test Stream)',
        filename = fname,
    )


## 13. Retrain Events – Agents vs Rolling Error (Test Stream)

In [ ]:
for name, results in {'MC Agent': results_mc, **results_td}.items():
    fname = name.lower().replace(' ', '_').replace('-', '_') + '_retrain_events.png'
    plot_retrain_events(
        results['rolling_error'],
        results['retrain_events'],
        title    = f'{name} – Rolling Error + Retrain Events (Test Stream)',
        filename = fname,
    )


## 14. Observations

**Setup:**
- Train stream: batch 0–{n_train-1} (70% data stream)
- Test stream : batch {n_train}–{n_total-1} (30% data stream)
- Warm-up     : `FULL_WINDOW_BATCHES` batches cuối train stream → điểm khởi đầu công bằng

**Câu hỏi cần trả lời sau khi chạy:**

*Learning curves (training):*
- TD agents hội tụ nhanh hơn hay chậm hơn MC Agent?
- Q-Learning có overestimate reward so với SARSA không?
- Double Q-Learning có ổn định hơn Q-Learning không (variance thấp hơn)?

*So sánh test stream:*
- Agent nào có prequential accuracy cao nhất?
- SARSA (on-policy) vs Q-Learning (off-policy) – cái nào tốt hơn trong non-stationary env?
- Expected SARSA có giảm variance so với SARSA không?
- Số lần retrain của TD agents so với MC Agent và Periodic baseline?

*Action distribution:*
- TD agents có action distribution khác MC Agent không?
- Agent nào dùng `partial_update` nhiều nhất?
- Agent nào thận trọng hơn (nhiều `no_action` hơn)?

→ Nếu TD tốt hơn MC: có thể do online update giúp adapt nhanh hơn với drift.  
→ Nếu MC tốt hơn TD: long-term returns giúp học policy tốt hơn immediate TD target.  
→ Thử tăng `N_TRAIN_EPISODES` lên 500 nếu chưa hội tụ.
